# DLM Beginner Tutorial — Setup & Orientation

This notebook verifies your environment and introduces the notation used throughout
the beginner series.

**Prerequisites:** Python fluency, basic probability (mean, variance, normal distribution).
No prior Bayesian or state-space modeling experience required.

**Series overview:**
- `B0_bayesian_primer.ipynb` *(optional)* — Bayesian modeling with PyMC from scratch
- `B1_dlm_intro.ipynb` — State space model equations and the Kalman filter
- `B2_local_level.ipynb` — Local-level model: filter, smoother, forecast
- `B3_local_linear_trend.ipynb` — Adding a slope state
- `B4_seasonal_models.ipynb` — Periodic components
- `B5_parameter_estimation.ipynb` — MLE and Bayesian estimation of V and W
- `B6_dlm_glm_connection.ipynb` *(optional)* — DLM as a generalization of regression

## 1. Environment check

In [ ]:
import sys
from pathlib import Path

# Add project root to path when running from notebooks/beginner/
project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt

print(f"Python  : {sys.version.split()[0]}")
print(f"NumPy   : {np.__version__}")
print(f"SciPy   : {scipy.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

# Engine imports — all should succeed
from engine.models import (
    DLMSpec, DLMSpecTV,
    make_local_level, make_local_linear_trend,
    make_seasonal_factor, combine,
)
from engine.filter import kalman_filter, kalman_filter_tv
from engine.smoother import rts_smoother
from engine.simulate import simulate
from engine.forecast import forecast_horizon

print("\nEngine imports: OK")

# Optional PyMC
try:
    import pymc as pm
    import arviz as az
    print(f"PyMC    : {pm.__version__}  (optional — used in B0 and B5)")
except ImportError:
    print("PyMC    : not installed (optional — skip B0, skip PyMC cells in B5)")

## 2. Notation

The Gaussian DLM uses West & Harrison (W&H) notation throughout this series.

**Model equations (W&H §2.2):**

$$
\begin{aligned}
y_t &= F_t \theta_t + v_t, \quad v_t \sim N(0, V_t) \quad \text{(observation equation)} \\
\theta_t &= G_t \theta_{t-1} + w_t, \quad w_t \sim N(0, W_t) \quad \text{(state equation)}
\end{aligned}
$$

| Symbol | Dimension | Meaning |
|--------|-----------|----------|
| $y_t$ | $(p,)$ | observed data at time $t$ |
| $\theta_t$ | $(d,)$ | latent state at time $t$ |
| $F_t$ | $(p, d)$ | observation matrix |
| $G_t$ | $(d, d)$ | state transition matrix |
| $V_t$ | $(p, p)$ | observation noise covariance |
| $W_t$ | $(d, d)$ | state evolution noise covariance |
| $m_t$ | $(d,)$ | filtered state mean $E[\theta_t \mid y_{1:t}]$ |
| $C_t$ | $(d, d)$ | filtered state covariance |
| $a_t$ | $(d,)$ | prior state mean $E[\theta_t \mid y_{1:t-1}]$ |
| $R_t$ | $(d, d)$ | prior state covariance |
| $e_t$ | $(p,)$ | innovation $y_t - F_t a_t$ |
| $Q_t$ | $(p, p)$ | innovation covariance $F_t R_t F_t' + V_t$ |
| $A_t$ | $(d, p)$ | Kalman gain $R_t F_t' Q_t^{-1}$ |

## 3. Reading guide

| Your background | Suggested path |
|-----------------|----------------|
| Never done Bayesian modeling | B0 → B1 → B2 → B3 → B4 → B5 |
| Know PyMC already | B1 → B2 → B3 → B4 → B5 |
| Know regression, want the GLM link | Add B6 after B5 |

After completing this series, open `notebooks/intermediate/00_setup.ipynb`.